<a href="https://colab.research.google.com/github/ms24191-PN/IDC-AQIupgrade/blob/main/Complete_data_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WITH NULL VALUES

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rohanrao/air-quality-data-in-india")

print("Path to dataset files:", path)

import os
print(os.listdir(path))

In [ ]:
import pandas as pd

file_path = os.path.join(path, 'city_day.csv')
data = pd.read_csv(file_path)
#data_final = data.dropna(subset=['AQI'])
#print (data_final.head())

print('DATA.SHAPE : ',"\n", data.shape, "\n", 'DATA.COLUMNS : ', "\n", data.columns, "\n",'DATA.HEAD : ', "\n",  data.head, "\n",'DATA.INFO', "\n",  data.info, "\n",'DATA.DESCRIBE',"\n", data.describe)

In [ ]:
print('NULL VALUES FOR RESPECTIVE COLUMNS :','\n', data.isnull().sum(),'\n', data.duplicated().sum())
print('DUPLICATED VALUES IN THE DATA :','\n',data.duplicated().sum())
print('NO OF UNIQUE CITIES : ','\n',data['City'].unique())
print('NO VALUES PER CITIES : ', '\n', data['City'].value_counts())
print('DTYPE OD DATE : ', '\n',data['Date'].dtype)

## Data feature addition

### Defining the Season function

In [ ]:
def get_season(month):
  #Standard seson month in NH are used here
  if month in [12,1,2]:
    return "Winter"
  elif month in [3,4,5]:
    return "Spring"
  elif month in [6,7,8]:
    return "Summer"
  else:
    return "Monsoon"

In [ ]:
data['Date'] = pd.to_datetime(data['Date'])
data['Year'] = data['Date'].dt.year
data['Month']= data['Date'].dt.month
data['Weekday'] = data['Date'].dt.day_name()
data['Day'] = data['Date'].dt.day
data['Season'] = data['Month'].apply(get_season)
data.head()
print(data.groupby('Season')['AQI'].mean())
data['Winter'] = data['Month'].isin([12,1,2]).astype(int) #Here 1 -> Winter month and 0 -> other month
print(data.groupby('Winter')['AQI'].mean())

# WITHOUT NULL VALUES

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rohanrao/air-quality-data-in-india")

print("Path to dataset files:", path)

import os
print(os.listdir(path))

In [ ]:
import pandas as pd

file_path = os.path.join(path, 'city_day.csv')
data = pd.read_csv(file_path)
data = data.dropna(subset=['AQI']) #Deleting the null values
#print (data_final.head())

print('DATA.SHAPE : ',"\n", data.shape, "\n", 'DATA.COLUMNS : ', "\n", data.columns, "\n",'DATA.HEAD : ', "\n",  data.head, "\n",'DATA.INFO', "\n",  data.info, "\n",'DATA.DESCRIBE',"\n", data.describe)

In [ ]:
print('NULL VALUES FOR RESPECTIVE COLUMNS :','\n', data.isnull().sum(),'\n', data.duplicated().sum())
print('DUPLICATED VALUES IN THE DATA :','\n',data.duplicated().sum()) #Checking the duplicate values
print('NO OF UNIQUE CITIES : ','\n',data['City'].unique())  #Checking the inconsistent city/station names.
print('NO VALUES PER CITIES : ', '\n', data['City'].value_counts())
print('DTYPE OD DATE : ', '\n',data['Date'].dtype)

## Data feature addition

### Defining the Season function

In [ ]:
def get_season(month):
  #Standard seson month in NH are used here
  if month in [12,1,2]:
    return "Winter"
  elif month in [3,4,5]:
    return "Spring"
  elif month in [6,7,8]:
    return "Summer"
  else:
    return "Monsoon"

In [ ]:
data['Date'] = pd.to_datetime(data['Date'])   #COnverting the dtype from object to datetime
data['Year'] = data['Date'].dt.year
data['Month']= data['Date'].dt.month     #Creating month data feature
data['Weekday'] = data['Date'].dt.day_name()   #Creating weekday data feature
data['Day'] = data['Date'].dt.day
data['Season'] = data['Month'].apply(get_season)     #Creating season  data feature
data.head()
print(data.groupby('Season')['AQI'].mean())
data['Winter'] = data['Month'].isin([12,1,2]).astype(int) #Here 1 -> Winter month and 0 -> other month
print(data.groupby('Winter')['AQI'].mean())      #Creating Winter indicator data feature

# Visualize city-wise AQI

## City-wise AQI

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

#let calculate the average of the year
avg_year_aqi = (data.groupby(['City','Year'])['AQI'].mean().reset_index())

cities = sorted(avg_year_aqi['City'].unique())
colors = sns.color_palette('husl', n_colors = len(cities) )

plt.figure(figsize = (16,8))

for city , color in zip(cities, colors):
  city_data = avg_year_aqi[avg_year_aqi['City'] == city].sort_values('Year')
  plt.plot(city_data['Year'], city_data['AQI'], marker = 'o', linewidth =2, markersize = 5, label= city, color = color)

plt.title('Annual aqi of the city-wise', fontsize = 16)
plt.xlabel('Year')
plt.ylabel('Average AQI')

plt.xticks(sorted(avg_year_aqi['Year'].unique()), rotation = 45)
plt.legend(title = 'City', bbox_to_anchor = (1.02,1), loc ='upper left')
plt.grid(True, alpha = 0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Count available AQI data points for each city
city_counts = (
    data.dropna(subset=['AQI'])
    .groupby('City')['AQI']
    .count()
    .sort_values(ascending=False)
)

# Select top 10 cities - most AQI data points
top_10_cities = city_counts.head(10).index.tolist()

# Filter data for selected cities
top_data = data[data['City'].isin(top_10_cities)].copy()

# 2. Calculate annual average AQI for each city
yearly_avg_aqi = (
    top_data.groupby(['City', 'Year'])['AQI']
    .mean()
    .reset_index()
)

# 3. Plot all 10 cities in a single graph
colors = sns.color_palette('husl', n_colors=len(top_10_cities))

plt.figure(figsize=(16, 8))

for city, color in zip(top_10_cities, colors):

    city_data = yearly_avg_aqi[
        yearly_avg_aqi['City'] == city
    ].sort_values('Year')

    plt.plot(
        city_data['Year'],
        city_data['AQI'],
        marker='o',
        linewidth=2,
        markersize=5,
        label=city,
        color=color
    )

plt.title('Annual Average AQI: Top 10 Cities by Data Availability',
          fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Annual Average AQI', fontsize=12)

plt.xticks(
    sorted(yearly_avg_aqi['Year'].unique()),
    rotation=45
)

plt.legend(
    title='City',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 4. Calculate overall average AQI for each selected city
city_avg_aqi = (
    top_data.groupby('City')
    .agg(
        Data_Points=('AQI', 'count'),
        Average_AQI=('AQI', 'mean')
    )
    .reset_index()
)

# Rank cities by average AQI
city_avg_aqi = city_avg_aqi.sort_values(
    'Average_AQI',
    ascending=False
).reset_index(drop=True)

city_avg_aqi['Average_AQI'] = city_avg_aqi['Average_AQI'].round(2)

# Display the complete ranking table
print("AQI Ranking of the Top 10 Cities")
display(city_avg_aqi)

# 5. Identify the highest and lowest average AQI cities
highest_aqi_city = city_avg_aqi.loc[
    city_avg_aqi['Average_AQI'].idxmax()
]

lowest_aqi_city = city_avg_aqi.loc[
    city_avg_aqi['Average_AQI'].idxmin()
]

# Create summary table
summary_table = pd.DataFrame([
    {
        'Category': 'Highest Average AQI',
        'City': highest_aqi_city['City'],
        'Average AQI': highest_aqi_city['Average_AQI'],
        'Data Points': highest_aqi_city['Data_Points']
    },
    {
        'Category': 'Lowest Average AQI',
        'City': lowest_aqi_city['City'],
        'Average AQI': lowest_aqi_city['Average_AQI'],
        'Data Points': lowest_aqi_city['Data_Points']
    }
])

print("Cities with the Highest and Lowest Average AQI")
display(summary_table)

### Pollutant correlations

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]

# Keep only pollutant columns available in the dataset
available_pollutants = [
    col for col in pollutants if col in data.columns
]

pollutant_data = data[available_pollutants].copy()


correlation_matrix = pollutant_data.corr(method='pearson')

# Plot the correlation heatmap
plt.figure(figsize=(13, 10))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
    vmin=-1,
    vmax=1,
    center=0,
    square=True
)

plt.title('Correlation Analysis Among Air Pollutants', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Yearly / Monthly Trends

Without averaging the month

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

cities = sorted(data['City'].unique())
years = sorted(data['Year'].unique())


year_colors = dict(
    zip(years, sns.color_palette('husl', n_colors=len(years)))
)

# Use a leap year - so that February 29th is valid
reference_year = 2020

# Create month and day from the ACTUAL Date column
#data['Month'] = data['Date'].dt.month
#data['Day'] = data['Date'].dt.day

# Create a common date using the actual month and day
data['Plot_Date'] = pd.to_datetime(
    str(reference_year) + '-' +
    data['Month'].astype(str).str.zfill(2) + '-' +
    data['Day'].astype(str).str.zfill(2)
)

for city in cities:

    # Select the city
    city_data = data[data['City'] == city].copy()

    plt.figure(figsize=(16, 8))

    for year in years:

        # Select the year
        year_data = city_data[city_data['Year'] == year].copy()

        if len(year_data) == 0:
            continue

        # Sort from January → December
        year_data = year_data.sort_values('Plot_Date')

        # Plot actual daily AQI
        plt.plot(
            year_data['Plot_Date'],
            year_data['AQI'],
            label=str(year),
            color=year_colors[year],
            linewidth=1
        )

    plt.title(f'Daily AQI Across Years - {city}')
    plt.xlabel('Month')
    plt.ylabel('AQI')

    # January to December ticks
    month_positions = pd.date_range(
        start=f'{reference_year}-01-01',
        end=f'{reference_year}-12-01',
        freq='MS'
    )

    month_labels = [
        'January', 'February', 'March', 'April',
        'May', 'June', 'July', 'August',
        'September', 'October', 'November', 'December'
    ]

    plt.xticks(
        month_positions,
        month_labels,
        rotation=45
    )

    # Start exactly at January and end at December
    plt.xlim(
        pd.Timestamp(f'{reference_year}-01-01'),
        pd.Timestamp(f'{reference_year}-12-31')
    )

    plt.legend(
        title='Year',
        bbox_to_anchor=(1.02, 1),
        loc='upper left'
    )

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

With averging the month

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create month from the original Date column
#data['Month'] = data['Date'].dt.month

# Calculate monthly AQI for each city and year
monthly_aqi = (
    data.groupby(['City', 'Year', 'Month'])['AQI']
    .mean()
    .reset_index()
)

# Get cities and years
cities = sorted(monthly_aqi['City'].unique())
years = sorted(monthly_aqi['Year'].unique())

# Create a different color for each year
year_colors = dict(
    zip(years, sns.color_palette('husl', n_colors=len(years)))
)

# Month names
month_labels = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

# Plot one graph for each city
for city in cities:

    city_data = monthly_aqi[
        monthly_aqi['City'] == city
    ].copy()

    plt.figure(figsize=(14, 7))

    for year in years:

        year_data = city_data[
            city_data['Year'] == year
        ].sort_values('Month')

        if len(year_data) == 0:
            continue

        plt.plot(
            year_data['Month'],
            year_data['AQI'],
            marker='o',
            linewidth=2,
            label=str(year),
            color=year_colors[year]
        )

    plt.title(f'Monthly AQI Across Years - {city}')
    plt.xlabel('Month')
    plt.ylabel('AQI')

    plt.xticks(
        range(1, 13),
        month_labels,
        rotation=45
    )

    plt.xlim(1, 12)

    plt.legend(
        title='Year',
        bbox_to_anchor=(1.02, 1),
        loc='upper left'
    )

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
print(data.columns)

# Model Traing

In [ ]:
data1 = data.copy()
# List of pollutant columns
pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]

# Replace missing values with the median of each pollutant
data1[pollutants] = data[pollutants].fillna(data[pollutants].median())

# Check missing values after imputation
print(data1[pollutants].isnull().sum())

# Display the first five rows
data1.head()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Define pollutant features (X)
pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]

# 2. Check that all required columns exist
required_columns = pollutants + ['AQI']

missing_columns = [
    col for col in required_columns
    if col not in data1.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# 3. Prepare features (X) and target (y)
X = data1[pollutants].copy()
y = pd.to_numeric(data1['AQI'], errors='coerce')

# Convert pollutant features to numeric
X = X.apply(pd.to_numeric, errors='coerce')

# 4. Remove rows where AQI (target) is missing
valid_rows = y.notna()

X = X.loc[valid_rows]
y = y.loc[valid_rows]

# 5. Split into training and testing sets (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# 6. Display the results
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nInput features:")
print(X_train.columns.tolist())

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create the Decision Tree model
model = make_pipeline(
    SimpleImputer(strategy='median'),
    DecisionTreeRegressor(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    )
)

# Train the model
model.fit(X_train, y_train)

# Predict AQI
y_pred = model.predict(X_test)

# Evaluate model performance
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create the model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

# Predict AQI
y_pred = model.predict(X_test)

# Evaluate the model
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
data2 = data.copy()
import pandas as pd
from sklearn.impute import KNNImputer

# Select the 12 pollutant columns
pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]

# Convert pollutant columns to numeric values
data2[pollutants] = data2[pollutants].apply(
    pd.to_numeric, errors='coerce'
)

# Create the KNN imputer
imputer = KNNImputer(n_neighbors=5, weights='distance')

# Fill missing values in pollutant columns
data2[pollutants] = imputer.fit_transform(data2[pollutants])

# Check missing values after imputation
print("Missing values after KNN imputation:")
print(data2[pollutants].isnull().sum())

# Display the updated data
data2.head()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Define pollutant features (X)
pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]

# 2. Check that all required columns exist
required_columns = pollutants + ['AQI']

missing_columns = [
    col for col in required_columns
    if col not in data2.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# 3. Prepare features (X) and target (y)
X = data2[pollutants].copy()
y = pd.to_numeric(data2['AQI'], errors='coerce')

# Convert pollutant features to numeric
X = X.apply(pd.to_numeric, errors='coerce')

# 4. Remove rows where AQI (target) is missing
valid_rows = y.notna()

X = X.loc[valid_rows]
y = y.loc[valid_rows]

# 5. Split into training and testing sets (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# 6. Display the results
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nInput features:")
print(X_train.columns.tolist())

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create the Decision Tree model
model = make_pipeline(
    SimpleImputer(strategy='median'),
    DecisionTreeRegressor(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    )
)

# Train the model
model.fit(X_train, y_train)

# Predict AQI
y_pred = model.predict(X_test)

# Evaluate model performance
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create the model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

# Predict AQI
y_pred = model.predict(X_test)

# Evaluate the model
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))